# Image Processing
This lesson introduces how to work with image data, extract basic image information, and perform common image-processing operations.

In this activity, we will learn how computers read, store, and process images.

We will use Python to:

- Open and display images
- Read image metadata
- Understand EXIF and GPS information
- Convert images between color spaces
- Apply thresholding, blurring, and edge detection
- Connect image processing with remote sensing and disaster response

We will mainly use two Python libraries:

- **Pillow (PIL)**: useful for opening images and reading image metadata such as EXIF (Exchangeable Image File Format) information ([Pillow](https://pillow.readthedocs.io/en/stable/index.html)).  
  Pillow provides base classes for representing image data, and some convenience helper functions and dictionaries.

- **OpenCV (`cv2`)**: useful for image processing tasks such as color-space conversion, filtering, thresholding, and edge detection  ([opencv](https://opencv.org/)).  
  OpenCV is a powerful image processing library. It is originally a C++ library, but we will be using the python bindings to call it in python. One important thing to remember about OpenCV is that **it represents image channels/Bands in BGR (Blue, Green, Red)** order (first channel is Blue), whereas most other image representations like **PIL represents image channels/Bands in RGB (Red, Green, Blue)** order (first channel is Red). Thus, you must make sure to switch up the order of channels when converting images between PIL and OpenCV.


In [ ]:
# Import libraries
import PIL    # Used for opening, reading, and working with image files using Pillow
import PIL.Image    # Used specifically to open and handle image objects
from PIL.ExifTags import TAGS, GPSTAGS    # Used to read and understand image metadata, including camera and GPS information
import cv2    # OpenCV library; used for image processing such as reading images, grayscale conversion, thresholding, and edge detection
import pathlib    # Used to work with file and folder paths in a clean and safe way
import numpy as np    # Used for numerical calculations and array operations, especially image pixel values
import pandas as pd    # Used for organizing data into tables, such as image metadata or GPS information
from matplotlib import pyplot as plt    # Used to display images, graphs, and figures
import os        # Used for working with folders, file paths, and checking files
import random    # Used for selecting something randomly, such as choosing one image from a folder
import pathlib   # Used for creating and working with file/folder paths
import rasterio   # Used for Rasterio reading, writing, and processing geospatial raster data


## Import images

First, we download and unzip a folder of sample images. Then we create a list of image files that Python can read.

In [ ]:
# Download the image dataset into a zip file named photos.zip.
!wget https://files.bwsi-remote-sensing.net/data/photos/BWSI-2019-MATF1-photos.zip -O photos.zip

In [ ]:
## Unzip the downloaded file.
# The -o option overwrites files if they already exist.
!unzip -o photos.zip

In [ ]:
# Set Image Folder Path
image_folder = pathlib.Path("BWSI-2019-MATF1-photos")

# Display the folder path
print("Image folder path:", image_folder)


In [ ]:
# Find compatibale image files
compatible_imgs = []

# Go through each file in the image folder
for file_name in os.listdir(image_folder):

    # Create full file path
    file_path = os.path.join(image_folder, file_name)

    # Check common image extensions
    if file_name.lower().endswith((".jpg", ".jpeg", ".png")):
        compatible_imgs.append(file_path)

# Display number of compatible images
print("Number of compatible images:", len(compatible_imgs))

# Display image list
print("Image list:")
for img_path in compatible_imgs:
    print(img_path)

In [ ]:
# Select one random image from the folder/list
random_img_path = np.random.choice(compatible_imgs)

# Open the selected image
random_img = PIL.Image.open(random_img_path)

# Display image path
print("Selected image:", random_img_path)

# Display the image
random_img

## Image Metadata (EXIF and GPS Metadata)

Image metadata is additional information stored with an image. It can include details such as image size, camera model, date/time, exposure settings, and sometimes location information.

- EXIF (Exchangeable Image File Format) is one common type of image metadata. It stores technical information about the image, such as camera `Make`, camera `Model`, image width, image height, and capture date/time.

- GPS information can also be stored inside EXIF metadata if the image contains location data. For example, GPS metadata may include latitude, longitude, and direction information.


- EXIF metadata often stores information using numeric codes. For example, EXIF tag `271` means camera `Make`, and tag `272` means camera `Model`. The `TAGS` dictionary (comes from the Pillow library) converts general EXIF numeric codes into readable names.  

- GPS metadata also uses numeric tag IDs. For example, GPS tag `1` means `GPSLatitudeRef`, and GPS tag `2` means `GPSLatitude`. The `GPSTAGS` dictionary (comes from the Pillow library) converts GPS-specific numeric codes into readable names.

- `TAGS` and `GPSTAGS` are reference dictionaries from the Pillow library. They only show the possible EXIF and GPS tag names that Pillow knows.


In [ ]:
# View EXIF Tag Dictionary
TAGS

In [ ]:
# View GPS Tag Dictionary
GPSTAGS

In [ ]:
# Read raw EXIF metadata from the selected random image
random_img_exif = random_img._getexif()
# Display raw EXIF metadata
random_img_exif


In [ ]:
# Read raw GPS metadata from the raw EXIF metadata
# 34853 is the EXIF tag ID for GPSInfo
gps_metadata = random_img_exif.get(34853) if random_img_exif is not None else None

# Display GPS metadata
gps_metadata

### Quick question

1. Do all images contain EXIF metadata? Yes/No? If yes, example?
2. If you take a photo with your phone, what information might be saved inside the image?

## Helper Functions Parsing EXIF and GPS Metadata

A helper function is a small function that performs one specific task.  

Example:
1. Convert fraction values into decimal numbers.
2. Convert GPS coordinates into decimal degrees.
3. Convert raw GPS metadata into readable GPS information.
4. Convert raw EXIF metadata into readable EXIF information.

In [ ]:
## Functions: Parse EXIF and GPS Metadata
# Convert GPS coordinates from DMS (Degrees, Minutes, Seconds) format to decimal degrees.
# For mapping, decimal degrees are easier to use.

# Function 1: Convert a fraction tuple into a decimal number
# Example: (1, 250) means 1/250 = 0.004
def divide_tuple(tup):
    try:
        return tup[0] / tup[1]
    except Exception:
        return None

# Function 2: Convert GPS coordinate from DMS to decimal degrees
# DMS means Degrees, Minutes, Seconds
def convert_gps_coord(gps_tuple):
    try:
        degrees = float(gps_tuple[0])
        minutes = float(gps_tuple[1])
        seconds = float(gps_tuple[2])

        decimal_degree = degrees + minutes / 60 + seconds / 3600
        return decimal_degree

    except Exception:
        return None

# Function 3: Make GPS metadata easier to read
def parse_gps(gps_dict):

    # Create an empty dictionary to store readable GPS information
    readable_gps = {}

    # If GPS information is missing, return the empty dictionary
    if gps_dict is None:
        return readable_gps

    # Go through each GPS item
    for key, value in gps_dict.items():

        # Convert GPS numeric code into readable GPS tag name
        text_key = GPSTAGS.get(key, key)

        # Convert GPS latitude and longitude from DMS to decimal degrees
        if text_key in ["GPSLatitude", "GPSLongitude"]:
            value = convert_gps_coord(value)

        # Convert bytes into readable hexadecimal text
        if isinstance(value, bytes):
            value = value.hex()

        # Store readable GPS information
        readable_gps[text_key] = value

    # Apply N/S/E/W direction to latitude and longitude
    if "GPSLatitude" in readable_gps and readable_gps.get("GPSLatitudeRef") == "S":
        readable_gps["GPSLatitude"] = -readable_gps["GPSLatitude"]

    if "GPSLongitude" in readable_gps and readable_gps.get("GPSLongitudeRef") == "W":
        readable_gps["GPSLongitude"] = -readable_gps["GPSLongitude"]

    return readable_gps


# Function 4: Make EXIF metadata easier to read
def parse_exif(exif_dict):

    # Create an empty dictionary to store readable EXIF information
    readable_exif = {}

    # If EXIF metadata is missing, return the empty dictionary
    if exif_dict is None:
        return readable_exif

    # Go through each EXIF item
    for key, value in exif_dict.items():

        # Convert EXIF numeric code into readable EXIF tag name
        text_key = TAGS.get(key, key)

        # Some EXIF values are stored as fraction tuples
        # Example: (1, 250) means 1/250 second
        if isinstance(value, tuple) and len(value) == 2:
            value = divide_tuple(value)

        # Convert bytes into readable hexadecimal text
        if isinstance(value, bytes):
            value = value.hex()

        # If the tag contains GPS information, parse it separately
        if text_key == "GPSInfo":
            readable_exif[text_key] = parse_gps(value)

        # Skip very large metadata fields
        elif text_key not in ["UserComment", "MakerNote"]:
            readable_exif[text_key] = value

    return readable_exif

In [ ]:
## Read and Display EXIF and GPS Metadata from the Selected Random Image

# Read raw EXIF metadata from the selected random image
random_img_exif = random_img._getexif()

# Convert raw EXIF metadata into readable format
readable_exif = parse_exif(random_img_exif)

# Create an empty list to store table rows
metadata_rows = []

# Go through each readable EXIF item
for key, value in readable_exif.items():

    # If GPS metadata exists, show each GPS item separately
    if key == "GPSInfo" and isinstance(value, dict):
        for gps_key, gps_value in value.items():
            metadata_rows.append({
                "Metadata_Type": "GPS",
                "Tag_Name": gps_key,
                "Value": gps_value
            })

    # Show regular EXIF metadata
    else:
        metadata_rows.append({
            "Metadata_Type": "EXIF",
            "Tag_Name": key,
            "Value": value
        })

# Convert metadata into a table
metadata_table = pd.DataFrame(metadata_rows)

# Display table
metadata_table

### Quick Question
Look at the parsed EXIF output. Can you find 5 items that you are familiar with?


In [ ]:
## Parse EXIF and GPS Metadata from All Compatible Images
# Iterate through all compatible images, parse EXIF and GPS metadata, and show in a table

metadata_rows = []

for image_path in compatible_imgs:

    # Open image
    img = PIL.Image.open(image_path)

    # Read raw EXIF metadata
    raw_exif = img._getexif()

    # Parse EXIF metadata into readable format
    readable_exif = parse_exif(raw_exif)

    # Start one row with basic image information
    row = {
        "Image_Name": os.path.basename(str(image_path)),
        "Image_Path": str(image_path),
        "Image_Format": img.format,
        "Image_Size": img.size,
        "Image_Width": img.width,
        "Image_Height": img.height
    }

    # Add parsed EXIF and GPS metadata
    for key, value in readable_exif.items():

        # Expand GPS metadata into separate columns
        if key == "GPSInfo" and isinstance(value, dict):
            for gps_key, gps_value in value.items():
                row["GPS_" + gps_key] = gps_value

        # Add regular EXIF metadata
        else:
            row[key] = value

    # Add this image's metadata to the list
    metadata_rows.append(row)

# Convert list of rows into a DataFrame table
metadata_table = pd.DataFrame(metadata_rows)

# Display the table
metadata_table

## Exercise: Create an Image Metadata Table for One Specific Image

Create a dataframe that stores important metadata for **one selected image** from the course photo folder.
Instead of processing all images, you will choose one image from `compatible_imgs` using an index number.

For example:
```python
image_index = 0
```
This means the **first image** in the list.
```python
image_index = 1
```
This means the **second image** in the list. 

Python indexing starts from **0**, not **1**.
You do not need to store every EXIF metadata field. Focus on useful information such as:

- file name
- file path
- image width and height
- image format
- camera make and model
- date and time
- GPS latitude and longitude, if available
- GPS altitude, if available

### Hints

1. Import the required libraries: `pathlib`, `PIL.Image`, and `pandas`.
2. Choose one image index: Set an index number, such as `image_index = 0`.
3. Remember Python indexing: Index `0` means the first image, index `1` means the second image.
4. Select the image path: Use `compatible_imgs[image_index]`.
5. Store the selected path: Save it in a variable named `img_path`.
6. Open the selected image: Use `PIL.Image.open(img_path)`.
7. Read raw EXIF metadata: Use `img._getexif()`.
8. Convert EXIF into readable form: Use `parse_exif()`.
9. Get GPS metadata: Use `exif_info.get("GPSInfo", {})`.
10. Create an empty list: Name it `metadata_list`.
11. Create one dictionary: Store metadata for the selected image.
12. Add basic image information: File name, file path, width, height, and format.
13. Add camera information: Camera make, camera model, and date/time.
14. Add GPS information: Latitude, longitude, latitude reference, longitude reference, and altitude.
15. Append the dictionary: Add it to `metadata_list`.
16. Create a dataframe: Use `pd.DataFrame(metadata_list)`.
17. Display the dataframe: Write `metadata_df`.

#### Important Note

GPS metadata is usually stored inside `GPSInfo`, so access it through `gps_info`, not directly from `exif_info`.

#### Optional Check

Before choosing an image index, you can print all image names with their index numbers:
```python
for i, img_path in enumerate(compatible_imgs):
    print(i, pathlib.Path(img_path).name)
```
Then choose the image index you want to analyze, example:
image_index = 2
```python
PIL.Image.open(compatible_imgs[1])
```

## Basic Image Processing
- We move from image metadata to image processing.
- In image processing, we use OpenCV to read an image as an array of pixel values.

## Understanding Image Color Spaces

A color space is a way to represent image colors using numbers. In image processing, choosing the correct color space is important because different tools and libraries may store color channels differently.

Key important things for image color spaces and conversion:

1. How to read an image using OpenCV.
2. Understand why OpenCV images may look strange in Matplotlib.
3. Convert the image from **BGR** to **RGB**.
4. Convert the image to **grayscale**.
5. Convert the image to **HSV** color space.
6. Display individual RGB and HSV channels.

**Important:** OpenCV reads color images as **BGR**: Blue, Green, Red. Matplotlib displays images as **RGB**: Red, Green, Blue. Therefore, we usually convert BGR to RGB before displaying an OpenCV image with Matplotlib.

In [ ]:
# Read a random image using OpenCV
# OpenCV reads color images in BGR format by default.

opencv_img_path = str(random_img_path)
raw_img = cv2.imread(opencv_img_path, cv2.IMREAD_COLOR)

# Check that OpenCV successfully loaded the image
if raw_img is None:
    raise FileNotFoundError("OpenCV could not read the selected image.")

# Resize the image for faster processing and easier display
lowres_dim = 1800
img_shape = raw_img.shape

img = cv2.resize(
    raw_img,
    (int((img_shape[1] / img_shape[0]) * lowres_dim), lowres_dim),
    interpolation=cv2.INTER_CUBIC
)

print("Selected image:", opencv_img_path)
print("Original image shape:", raw_img.shape)
print("Resized image shape:", img.shape)

In [ ]:
# Display the OpenCV image without color conversion
# The colors may look incorrect because OpenCV stores the image as BGR,
# And when OpenCV uses Matplotlib, Matplotlib assumes the image is RGB. That's why it displays unexpected true color.
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.title("OpenCV Image Displayed Without BGR-to-RGB Conversion")
plt.axis("off")
plt.show()


In [ ]:
# Convert the OpenCV BGR image into different color spaces
# So when we display an OpenCV image using Matplotlib, we should convert BGR to RGB.
RGB_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)    # Correct color order for Matplotlib

# Some OpenCV functions use the American spelling "GRAY" instead of "GREY".
grey_img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)  # Single-channel grayscale image
HSV_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)    # Hue, Saturation, Value color space

print("RGB image shape:", RGB_img.shape)
print("Grayscale image shape:", grey_img.shape)
print("HSV image shape:", HSV_img.shape)

In [ ]:
# Display the corrected true color RGB image
# This image should show natural true colors because it was converted from BGR to RGB.
plt.figure(figsize=(10, 8))
plt.imshow(RGB_img)
plt.title("Corrected True Color RGB Image")
plt.axis("off")
plt.show()

In [ ]:
# Separate and display the RGB color channels
# Each channel shows the intensity of one color component.

red_channel = RGB_img[:, :, 0]
green_channel = RGB_img[:, :, 1]
blue_channel = RGB_img[:, :, 2]

fig, axes = plt.subplots(3, 1, figsize=(10, 8))

axes[0].imshow(red_channel, cmap="Reds")
axes[0].set_title("Red Channel", fontsize=14)
axes[0].axis("off")

axes[1].imshow(green_channel, cmap="Greens")
axes[1].set_title("Green Channel", fontsize=14)
axes[1].axis("off")

axes[2].imshow(blue_channel, cmap="Blues")
axes[2].set_title("Blue Channel", fontsize=14)
axes[2].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Plot histograms for the red, green, and blue channels
# A histogram shows how often each pixel value occurs.
# Pixel values usually range from 0 to 255.

fig, axes = plt.subplots(3, 1, figsize=(10, 8))

axes[0].hist(red_channel.flatten(), bins=256, range=[0, 256], color="r")
axes[0].set_title("Red Channel Histogram")
axes[0].set_xlabel("Pixel value")
axes[0].set_ylabel("Frequency")

axes[1].hist(green_channel.flatten(), bins=256, range=[0, 256], color="g")
axes[1].set_title("Green Channel Histogram")
axes[1].set_xlabel("Pixel value")
axes[1].set_ylabel("Frequency")

axes[2].hist(blue_channel.flatten(), bins=256, range=[0, 256], color="b")
axes[2].set_title("Blue Channel Histogram")
axes[2].set_xlabel("Pixel value")
axes[2].set_ylabel("Frequency")

plt.tight_layout()
plt.show()

### HSV Color Space

HSV stands for **Hue, Saturation, and Value**.

- **Hue** represents the color type, such as red, green, or blue.
- **Saturation** represents color intensity. Low saturation looks gray, and high saturation looks colorful.
- **Value** represents brightness. Low value is dark, and high value is bright.

HSV is useful for image-processing tasks where we want to separate color information from brightness.

![alt text](https://upload.wikimedia.org/wikipedia/commons/3/33/HSV_color_solid_cylinder_saturation_gray.png)

In [ ]:
# Extract the Hue channel from the HSV image
hue_channel = HSV_img[:, :, 0]

# Extract the Saturation channel from the HSV image
saturation_channel = HSV_img[:, :, 1]

# Extract the Value/Brightness channel from the HSV image
value_channel = HSV_img[:, :, 2]

# Display the three HSV channels
fig, axes = plt.subplots(3, 1, figsize=(10, 8))

axes[0].imshow(hue_channel, cmap="Greys_r")
axes[0].set_title("Hue Channel", fontsize=14)
axes[0].axis("off")

axes[1].imshow(saturation_channel, cmap="Greys_r")
axes[1].set_title("Saturation Channel", fontsize=14)
axes[1].axis("off")

axes[2].imshow(value_channel, cmap="Greys_r")
axes[2].set_title("Value / Brightness Channel", fontsize=14)
axes[2].axis("off")

plt.tight_layout()
plt.show()

### Displaying HSV Channels as an Image

Matplotlib assumes that a three-channel image is in RGB order. If we display an HSV image directly using `plt.imshow()`, Matplotlib treats the Hue channel as Red, the Saturation channel as Green, and the Value channel as Blue.

This does not show a true HSV visualization. It simply shows what happens when HSV values are displayed as if they were RGB values.

In [ ]:
# Display HSV channels as if they were RGB channels
# This looks unusual because the channels do not represent Red, Green, and Blue.

plt.figure(figsize=(10, 8))
plt.imshow(HSV_img)
plt.title("HSV Channels Displayed as RGB", fontsize=14)
plt.axis("off")
plt.show()

## Image processing: Convolution Filters

Convolution is an image-processing operation where a small matrix, called a **kernel**, moves across an image. At each location, the kernel is combined with nearby pixel values to create a new output pixel.

![convolution_example](https://upload.wikimedia.org/wikipedia/commons/1/19/2D_Convolution_Animation.gif)

Different kernels produce different effects:

- **Identity:** keeps the image mostly unchanged.
- **Edge detection:** highlights rapid changes in pixel values.
- **Sharpening:** makes boundaries and details appear stronger.
- **Box blur:** smooths the image using a simple average.
- **Gaussian blur:** smooths the image using a weighted average, giving more weight to nearby pixels.
 
Here is an input image:

![input image](https://upload.wikimedia.org/wikipedia/commons/5/50/Vd-Orig.png)

Then, here are some types of kernels that can be applied and their effects:

### Identity
![identity kernel](https://wikimedia.org/api/rest_v1/media/math/render/svg/1fbc763a0af339e3a3ff20af60a8a993c53086a7)

![input image](https://upload.wikimedia.org/wikipedia/commons/5/50/Vd-Orig.png)
### Edge detection
![edge_kernel](https://wikimedia.org/api/rest_v1/media/math/render/svg/f9de5913c98629f30efb20b8868e096f202b626c)

![edge_result](https://upload.wikimedia.org/wikipedia/commons/2/20/Vd-Rige1.png)
### Sharpen
![sharpen_kernel](https://wikimedia.org/api/rest_v1/media/math/render/svg/beb8b9a493e8b9cf5deccd61bd845a59ea2e62cc)

![sharpen_result](https://upload.wikimedia.org/wikipedia/commons/4/4e/Vd-Sharp.png)
### Box Blur 
![box_blur_kernel](https://wikimedia.org/api/rest_v1/media/math/render/svg/91256bfeece3344f8602e288d445e6422c8b8a1c)

![box blur result](https://upload.wikimedia.org/wikipedia/commons/0/04/Vd-Blur2.png)
### Gaussian blur (3x3) 
![gauss kernel 3x3](https://wikimedia.org/api/rest_v1/media/math/render/svg/ca9c0da52fe7818783942b06aac9cf396ae628bf)

![gauss result 3x3](https://upload.wikimedia.org/wikipedia/commons/2/28/Vd-Blur1.png)
### Gaussian blur (5x5) 
![gauss kernel 5x5](https://wikimedia.org/api/rest_v1/media/math/render/svg/f91401a3e97428f14862afa1c781c55f4157580b)

![gauss result 5x5](https://upload.wikimedia.org/wikipedia/commons/0/04/Vd-Blur_Gaussian_5x5.png)

OpenCV supports custom convolution kernels using `cv2.filter2D()`. It also provides built-in functions for common filters, such as `cv2.GaussianBlur()`.

### Apply Sharpening and Blur Filters

In the next examples, we apply two common filters:

1. A **sharpening filter** to enhance edges and details.
2. A **Gaussian blur filter** to smooth the image and reduce noise.

### Quick Question:

Before running the sharpening code, what do you expect to happen?


In [ ]:
# Apply a sharpening convolution filter
# This kernel increases the center pixel value and subtracts nearby pixels.
# The result makes edges and details appear sharper.

sharpen_kernel = np.array([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])

# The -1 value keeps the output image depth the same as the input image.
sharpened_img = cv2.filter2D(RGB_img, -1, sharpen_kernel)

fig, axes = plt.subplots(1, 2, figsize=(10, 8))

axes[0].imshow(RGB_img)
axes[0].set_title("Original True color RGB Image", fontsize=14)
axes[0].axis("off")

axes[1].imshow(sharpened_img)
axes[1].set_title("Sharpened True color RGB Image", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Apply Gaussian blur to smooth the image.
# It alos reduce noise and avoid too many false edges..
# Blurring is often used before thresholding or edge detection.
# Kernel size controls the neighborhood size.
# Sigma controls the strength of the smoothing.

gauss_blur_kernel_size = (13, 13)
gauss_blur_kernel_sigma = 19

blurred_RGB_img = cv2.GaussianBlur(
    RGB_img,
    gauss_blur_kernel_size,
    gauss_blur_kernel_sigma
)

fig, axes = plt.subplots(1, 2, figsize=(10, 8))

axes[0].imshow(RGB_img)
axes[0].set_title("Original True Color RGB Image", fontsize=14)
axes[0].axis("off")

axes[1].imshow(blurred_RGB_img)
axes[1].set_title("Gaussian True Color RGB Blurred Image", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
plt.show()

## Thresholding

Thresholding converts a grayscale image into a binary image. In a binary image, pixels usually become either **0** for black or **255** for white.

Thresholding is useful for separating objects or regions of interest from the background.

Common thresholding methods include:

- **Simple binary thresholding:** manually choose one threshold value.
- **Adaptive thresholding:** calculate local threshold values using nearby pixels.
- **Otsu thresholding:** automatically choose a threshold value from the image histogram.

In [ ]:
# Display the grayscale image before thresholding.
plt.figure(figsize=(10, 8))
plt.imshow(grey_img, cmap="Greys_r")
plt.title("Grayscale Image Before Thresholding")
plt.axis("off")
plt.show()

In [ ]:
# Simple binary thresholding.
# OpenCV uses 128 as the cutoff threshold here.
# Pixel values greater than 128 are converted to 255/white.
# Pixel values less than or equal to 128 are converted to 0/black.

ret, grey_thresh = cv2.threshold(grey_img, 128, 255, cv2.THRESH_BINARY)

print("Threshold value:", ret)

plt.figure(figsize=(10, 8))
plt.imshow(grey_thresh, cmap="Greys_r")
plt.title("Simple Binary Thresholding")
plt.axis("off")
plt.show()

In [ ]:
# Check the thresholded image array
# After binary thresholding, the image should contain only 0 and 255 values.

grey_thresh

In [ ]:
# Check the unique pixel values in the thresholded image.
# For binary thresholding, we expect only 0 and 255.

np.unique(grey_thresh)

### Adaptive Thresholding

Simple thresholding uses one fixed threshold value for the whole image. Adaptive thresholding calculates different threshold values for different parts of the image.

This is helpful when lighting is uneven.

Two common adaptive thresholding methods are:

- `cv2.ADAPTIVE_THRESH_MEAN_C`: threshold value is based on the mean of the neighborhood.
- `cv2.ADAPTIVE_THRESH_GAUSSIAN_C`: threshold value is based on a weighted Gaussian neighborhood.

Important parameters:

- **block size:** number of nearby pixels used to calculate the local threshold.
- **C:** a constant subtracted from the calculated threshold.

In [ ]:
# Adaptive thresholding using the mean of a local neighborhood
grey_thresh_mean = cv2.adaptiveThreshold(
    grey_img,
    255,
    cv2.ADAPTIVE_THRESH_MEAN_C,
    cv2.THRESH_BINARY,
    301,
    5
)

plt.figure(figsize=(10, 8))
plt.imshow(grey_thresh_mean, cmap="Greys_r")
plt.title("Adaptive Mean Thresholding")
plt.axis("off")
plt.show()

In [ ]:
# Adaptive thresholding using a Gaussian-weighted local neighborhood
grey_thresh_gaussian = cv2.adaptiveThreshold(
    grey_img,
    255,
    cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
    cv2.THRESH_BINARY,
    501,
    10
)

plt.figure(figsize=(10, 8))
plt.imshow(grey_thresh_gaussian, cmap="Greys_r")
plt.title("Adaptive Gaussian Thresholding")
plt.axis("off")
plt.show()

In [ ]:
# Otsu thresholding automatically chooses a threshold value from the image histogram.
ret, grey_thresh_otsu = cv2.threshold(
    grey_img,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print("Otsu threshold value:", ret)

plt.figure(figsize=(10, 8))
plt.imshow(grey_thresh_otsu, cmap="Greys_r")
plt.title("Otsu Thresholding")
plt.axis("off")
plt.show()

In [ ]:
# Applying Gaussian blur before thresholding can reduce noise.
blurred_img = cv2.GaussianBlur(grey_img, (5, 5), 0)

plt.figure(figsize=(10, 8))
plt.imshow(blurred_img, cmap="Greys_r")
plt.title("Blurred Grayscale Image")
plt.axis("off")
plt.show()

In [ ]:
# Otsu thresholding after Gaussian blur.
# This often gives a cleaner thresholded result.
# The advantage is that the threshold value in Otsu thresholding is automatically chosen.
# This is useful when we do not know the best threshold manually.

ret, grey_thresh_blur_otsu = cv2.threshold(
    blurred_img,
    0,
    255,
    cv2.THRESH_BINARY + cv2.THRESH_OTSU
)

print("Otsu threshold value after blur:", ret)

plt.figure(figsize=(10, 8))
plt.imshow(grey_thresh_blur_otsu, cmap="Greys_r")
plt.title("Otsu Thresholding After Gaussian Blur")
plt.axis("off")
plt.show()

## Edge Detection

Edge detection identifies locations where pixel values change sharply. These changes often represent object boundaries, lines, or texture changes.

In this section, we use the **Canny edge detector**, a common method for detecting edges in images. Before edge detection, smoothing can help reduce noise.

In [ ]:
# Define a function for edge detection
def get_edges(img):
    # Apply Gaussian blur to reduce noise
    img = cv2.GaussianBlur(img, (11, 11), 1)

    # Apply bilateral filter for additional noise reduction while preserving edges
    img = cv2.bilateralFilter(img, 5, 20, 20)

    # Compute edges using Canny edge detection
    edges = cv2.Canny(img, 30, 60)

    return edges

In [ ]:
# Detect edges from the grayscale image
plt.figure(figsize=(10, 8))
plt.imshow(get_edges(grey_img), cmap="Greys")
plt.title("Edges from Grayscale Image")
plt.axis("off")
plt.show()

In [ ]:
# Detect edges from the grayscale image
plt.figure(figsize=(10, 8))
plt.imshow(get_edges(grey_img), cmap="Greys")
plt.title("Edges from Grayscale Image")
plt.axis("off")
plt.show()

In [ ]:
# Detect edges from the Value/Brightness channel of the HSV image
plt.figure(figsize=(10, 8))
plt.imshow(get_edges(value_channel), cmap="Greys")
plt.title("Edges from HSV Value Channel")
plt.axis("off")
plt.show()

### Recap questions and reflection

1. How many channels does an RGB image have?
2. What does EXIF metadata store?
3. What does edge detection find?
4. Which image-processing part is most interesting for you?
5. What is one new thing you learned today?


## Exercise: Apply Image-Processing Techniques to Remote Sensing Images

Using the image-processing techniques learned today, apply them to analyze satellite, aerial, or drone imagery.

For example, you can use **thresholding** to isolate a burned area or use **edge detection** to identify the boundary or front of a fire.

In this lecture, the provided Landsat 8 image was ***acquired on July 27, 2024; location: Park Fire in Northern California***, and the composite 7 Bands are:

- Band 1 = Coastal Aerosol
- Band 2 = Blue
- Band 3 = Green
- Band 4 = Red
- Band 5 = Near-Infrared (NIR)
- Band 6 = Shortwave Infrared 1 (SWIR 1)
- Band 7 = Shortwave Infrared 2 (SWIR 2)

### What You Should Do:

1. Open the Landsat 8 composite image using `rasterio`.
2. Read the required bands:
   - Blue = Band 2
   - Green = Band 3
   - Red = Band 4
   - NIR = Band 5
   - SWIR 2 = Band 7
3. Create and display the original true-color RGB image.
4. Calculate the Normalized Burn Ratio, or NBR.
5. Normalize the NBR image to 0–255 for OpenCV processing.
6. Apply Gaussian blur to reduce noise.
7. Apply thresholding to identify possible burned areas.
8. Apply edge detection to identify the fire boundary.
9. Create a burn-area overlay on the original RGB image.
10. Compare spectral values for burned and unburned areas.
11. Display and save the final maps and graph.
12. Optional: Show the spectral reflectance curve between burned and unburned area.

### Hints:

1. Import the required libraries: `rasterio`, `numpy`, `matplotlib.pyplot`, and `cv2`.
2. Set the Landsat image path: `image_path = "Landsat8_Satellite_Image.tif"`.
3. Open the image using `rasterio.open(image_path)`.
4. Read the required bands: Blue = Band 2, Green = Band 3, Red = Band 4, NIR = Band 5, SWIR 2 = Band 7.
5. Convert the bands to float before calculation.
6. Create the true-color image by stacking Red, Green, and Blue.
7. Calculate NBR: `NBR = (NIR - SWIR) / (NIR + SWIR + 1e-10)`.
8. Normalize NBR to 0–255 for OpenCV processing.
9. Apply Gaussian blur to reduce noise.
10. Use thresholding to isolate possible burned areas.
11. Try different threshold values, such as `100`, `120`, `140`, or `160`.
12. Use `cv2.THRESH_BINARY_INV` because burned areas usually have lower NBR values.
13. Apply Canny edge detection to find the burned-area boundary.
14. Create an overlay by marking burned pixels in red on the RGB image.
15. Compare burned and unburned pixels using mean values from Blue, Green, Red, NIR, and SWIR 2 bands.
16. Display the maps and spectral graph.
17. Briefly explain what the burn-area map, edge map, overlay, and spectral graph show.

### Remember

- OpenCV reads images as **BGR**, but Matplotlib displays images as **RGB**.
- Use **grayscale** images for thresholding and edge detection.
- Use **Gaussian blur** before thresholding or edge detection to reduce noise.
- Use **thresholding** to separate regions.
- Use **edge detection** to identify boundaries.
- Different images may need different parameter values.

### Example Other Applications

| Application | Useful Method | Purpose |
|---|---|---|
| Burned-area mapping | Thresholding or NBR | Separate burned and unburned areas |
| Fire-front detection | Edge detection | Find fire boundary |
| Vegetation analysis | RGB/HSV channels | Identify vegetation patterns |
| Water-body extraction | Thresholding | Separate water from land |
| Image enhancement | Blur or sharpening | Improve visual interpretation |


### Quick Check and Interpretation

Why do we compare burned and unburned spectral values?

Answer: To see how their spectral patterns are different.  
The spectral profile compares burned and unburned pixels (used normalized pixel values) from Landsat 8 image. Burned areas usually show lower NIR values and higher SWIR values because fire removes healthy vegetation and exposes ash, dry soil, or burned material.

How can image processing help in disaster response?

Answer: Image processing helps us find disaster-affected areas quickly from images.  
It can show where fire, flood, or damage happened, so responders can plan faster and help people more effectively.


